# Treinamento de Modelos ML XGBoost e CatBoost

Este notebook executa o fluxo completo de treinamento, otimização e validação de modelos baseados em árvore.

### Importação das Bibliotecas
Importa as dependências do ecossistema Scikit-Learn, Optuna para otimização e as bibliotecas base.

In [6]:
import pandas as pd
import numpy as np
import warnings
import os
import joblib
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

os.makedirs('models', exist_ok=True)

## 1. Funções Utilitárias

In [7]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

def evaluate(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    r = rmse(y_true, y_pred)
    return {'MAE': round(mae, 2), 'RMSE': round(r, 2)}

### Preparação e Tratamento de Nulos
Isola-se a variável alvo (`Revenue`) e aplica-se o Forward Fill seguido de Backward Fill temporal para preencher lacunas sem causar vazamento de dados do futuro.

In [8]:
def load_data(csv_path):
    df = pd.read_csv(csv_path, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
    drop_cols = ['Date']
    target = 'Revenue'
    feature_cols = [c for c in df.columns if c not in drop_cols + [target] and pd.api.types.is_numeric_dtype(df[c])]
    
    # Preenchimento de NaN: forward fill depois back fill
    df[feature_cols] = df[feature_cols].ffill().bfill().fillna(0)
    
    X = df[feature_cols].values
    y = df[target].values
    return X, y, feature_cols

## 2. Carregamento dos Dados

### Carregamento das Partições
Lê as três frentes de dados independentes (Produto, País, Cliente) e cria as matrizes de características (`X`) e a série alvo (`y`).

In [9]:
datasets = {
    'Produto': 'data/product.csv',
    'País': 'data/country.csv',
    'Cliente': 'data/customer.csv'
}

data = {}
for name, path in datasets.items():
    X, y, feat_cols = load_data(path)
    data[name] = {'X': X, 'y': y, 'feature_cols': feat_cols}
    print(f"  {name}: total_samples={X.shape[0]}")

  Produto: total_samples=454
  País: total_samples=529
  Cliente: total_samples=738


## 3. XGBoost (10-Fold Time Series Split)

### Otimização e Treinamento do XGBoost
Aplica validação cruzada (`TimeSeriesSplit` com 10 Folds). O Optuna utiliza os últimos 15% do treino atual como validação interna para achar hiperparâmetros ótimos.
O modelo do último fold é exportado (`.pkl`).

In [10]:
N_SPLITS = 10
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
results_xgb = {}

for name, d in data.items():
    print(f"\nOtimizando XGBoost para a série: {name}")
    X, y = d['X'], d['y']
    
    fold_metrics = []
    last_fold_preds = None
    last_fold_n = 0
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]
        
        inner_val_size = max(int(len(X_train) * 0.15), 1)
        X_train_inner, y_train_inner = X_train[:-inner_val_size], y_train[:-inner_val_size]
        X_inner_val, y_inner_val = X_train[-inner_val_size:], y_train[-inner_val_size:]
        
        def objective(trial):
            params = {
                'n_estimators': 200,
                'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
                'max_depth': trial.suggest_int('max_depth', 3, 7),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'random_state': 42,
                'verbosity': 0
            }
            model = XGBRegressor(**params)
            model.fit(X_train_inner, y_train_inner, eval_set=[(X_inner_val, y_inner_val)], verbose=False)
            preds = np.maximum(model.predict(X_inner_val), 0)
            return rmse(y_inner_val, preds)
            
        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=10)
        
        best_params = study.best_params
        best_params['n_estimators'] = 200
        best_params['random_state'] = 42
        best_params['verbosity'] = 0
        
        best_model = XGBRegressor(**best_params)
        best_model.fit(X_train, y_train, verbose=False)
        
        y_pred = np.maximum(best_model.predict(X_val), 0)
        metrics = evaluate(y_val, y_pred)
        fold_metrics.append(metrics)
        
        if fold == N_SPLITS - 1:
            joblib.dump(best_model, f'models/result_pkl/xgb_{name}.pkl')
            last_fold_preds = y_pred.tolist()
            last_fold_n = len(y_pred)
            
    avg_mae = np.mean([m['MAE'] for m in fold_metrics])
    avg_rmse = np.mean([m['RMSE'] for m in fold_metrics])
    results_xgb[name] = {
        'MAE': avg_mae,
        'RMSE': avg_rmse,
        'valores_previstos': last_fold_preds,
        'periodos_previstos': last_fold_n
    }
    print(f"  {name} XGBoost Médio (10 Folds): MAE={avg_mae:.2f}, RMSE={avg_rmse:.2f}")


Otimizando XGBoost para a série: Produto
  Produto XGBoost Médio (10 Folds): MAE=97.19, RMSE=119.92

Otimizando XGBoost para a série: País
  País XGBoost Médio (10 Folds): MAE=2276.79, RMSE=2945.49

Otimizando XGBoost para a série: Cliente
  Cliente XGBoost Médio (10 Folds): MAE=93.29, RMSE=218.07


## 4. CatBoost (10-Fold Time Series Split)

### Otimização e Treinamento do CatBoost
Semelhante ao XGBoost, realiza a busca estruturada por `learning_rate` e `depth` otimizados para evitar sobreajuste temporal.

In [11]:
results_cat = {}

for name, d in data.items():
    print(f"\nOtimizando CatBoost para a série: {name}")
    X, y = d['X'], d['y']
    
    fold_metrics = []
    last_fold_preds = None
    last_fold_n = 0
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]
        
        inner_val_size = max(int(len(X_train) * 0.15), 1)
        X_train_inner, y_train_inner = X_train[:-inner_val_size], y_train[:-inner_val_size]
        X_inner_val, y_inner_val = X_train[-inner_val_size:], y_train[-inner_val_size:]
        
        def objective(trial):
            params = {
                'iterations': 200,
                'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
                'depth': trial.suggest_int('depth', 4, 6),
                'random_seed': 42,
                'verbose': 0
            }
            model = CatBoostRegressor(**params)
            model.fit(X_train_inner, y_train_inner, eval_set=(X_inner_val, y_inner_val), verbose=0)
            preds = np.maximum(model.predict(X_inner_val), 0)
            return rmse(y_inner_val, preds)
            
        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=10)
        
        best_params = study.best_params
        best_params['iterations'] = 200
        best_params['random_seed'] = 42
        best_params['verbose'] = 0
        
        best_model = CatBoostRegressor(**best_params)
        best_model.fit(X_train, y_train, verbose=0)
        
        y_pred = np.maximum(best_model.predict(X_val), 0)
        metrics = evaluate(y_val, y_pred)
        fold_metrics.append(metrics)
        
        if fold == N_SPLITS - 1:
            joblib.dump(best_model, f'models/result_pkl/cat_{name}.pkl')
            last_fold_preds = y_pred.tolist()
            last_fold_n = len(y_pred)
            
    avg_mae = np.mean([m['MAE'] for m in fold_metrics])
    avg_rmse = np.mean([m['RMSE'] for m in fold_metrics])
    results_cat[name] = {
        'MAE': avg_mae,
        'RMSE': avg_rmse,
        'valores_previstos': last_fold_preds,
        'periodos_previstos': last_fold_n
    }
    print(f"  {name} CatBoost Médio (10 Folds): MAE={avg_mae:.2f}, RMSE={avg_rmse:.2f}")


Otimizando CatBoost para a série: Produto
  Produto CatBoost Médio (10 Folds): MAE=94.20, RMSE=117.33

Otimizando CatBoost para a série: País
  País CatBoost Médio (10 Folds): MAE=2339.87, RMSE=3046.38

Otimizando CatBoost para a série: Cliente
  Cliente CatBoost Médio (10 Folds): MAE=99.23, RMSE=237.82


### Exportação dos Resultados
Consolida a média de MAE e RMSE de todas as séries avaliadas, junto com os valores previstos do último fold, em um arquivo CSV final.

In [12]:
import os
os.makedirs('resultados', exist_ok=True)

rows = []
for name, m in results_xgb.items():
    rows.append({
        'serie': name,
        'periodos_previstos': m['periodos_previstos'],
        'modelo': 'XGBoost',
        'valores_previstos': m['valores_previstos'],
        'MAE': round(m['MAE'], 2),
        'RMSE': round(m['RMSE'], 2)
    })
for name, m in results_cat.items():
    rows.append({
        'serie': name,
        'periodos_previstos': m['periodos_previstos'],
        'modelo': 'CatBoost',
        'valores_previstos': m['valores_previstos'],
        'MAE': round(m['MAE'], 2),
        'RMSE': round(m['RMSE'], 2)
    })

df_results = pd.DataFrame(rows)
df_results.to_csv('resultados/resultados_ML.csv', index=False)
print('Resultados salvos em resultados/resultados_ML.csv')
df_results

Resultados salvos em resultados/resultados_ML.csv


,serie,periodos_previstos,modelo,valores_previstos,MAE,RMSE
0,Produto,41,XGBoost,"[205.82943725585938, 178.0882110595703, 184.31...",97.20,119.92
1,País,48,XGBoost,"[23350.34765625, 29820.1484375, 21600.18945312...",2276.79,2945.49
2,Cliente,67,XGBoost,"[16.68895721435547, 16.68895721435547, 16.6889...",93.29,218.07
3,Produto,41,CatBoost,"[182.96686205955876, 150.26106712490994, 163.3...",94.20,117.33
4,País,48,CatBoost,"[24586.114293822007, 28904.471689058788, 21895...",2339.87,3046.38
5,Cliente,67,CatBoost,"[46.131866349864026, 42.55730609197411, 22.590...",99.23,237.82
